# Listing Search Evaluation Bench

Inspect ranking failures with explicit graded judgments and per-query reports. The authored English/Russian dataset contains 8 fictional candidates, 14 queries and 112 candidate judgments. Budget points are invented unitless numbers. No live listings or market claims are included.

Prepared by [NovAsia Estate](https://novasia.estate/). [Code, complete judgments and tests](https://github.com/novasia-estate/open-property-lab/tree/main/listing-search-evaluation). MIT.


In [ ]:
from pathlib import Path
from html import escape
from IPython.display import HTML, display
import json
import search_eval
from search_eval import evaluate, load
root = Path(search_eval.__file__).parent
def summary_html(result):
    names = ['system', 'answerable_queries', 'precision_at_k', 'recall_at_k', 'ndcg_at_k', 'reciprocal_rank_at_k', 'correct_empty_results']
    def format_value(x): return f'{x:.3f}' if type(x) is float else str(x)
    table = '<table><tr>'+''.join('<th>'+escape(n)+'</th>' for n in names)+'</tr>'
    for row in result['summary']:
        table += '<tr>'+''.join('<td>'+escape(format_value(row[n]))+'</td>' for n in names)+'</tr>'
    return table+'</table>'


## Compare disclosed baselines

`lexical_only` matches literal NFKC/case-folded Unicode tokens. `explicit_filters_plus_lexical` additionally receives author-supplied constraints; it does not infer them from the query. The comparison therefore does not measure natural-language understanding or a trained model. Unknown pet policy remains unknown.

Precision divides relevant retrieved candidates by the declared cutoff even if fewer results are returned. Answerable-query means are kept separate from no-relevant-query abstention. Higher nDCG indicates a better order under these authored grades.


In [ ]:
cutoff = 3
result = evaluate(root, k=cutoff)
display(HTML(summary_html(result)))

system,answerable_queries,precision_at_k,recall_at_k,ndcg_at_k,reciprocal_rank_at_k,correct_empty_results
lexical_only,12,0.444,0.861,0.899,0.917,0
explicit_filters_plus_lexical,12,0.556,1.000,1.000,1.000,2


## Inspect an individual failure

Change `query_id` to any identifier in `queries.json` and rerun. The report retains the explicit constraints, ordered candidates, missed relevant IDs and all metrics. The two no-match queries should produce empty results. Matching the word `pets` is not sufficient to distinguish permission, prohibition and unknown policy.


In [ ]:
query_id = 'pets-metro-ru'
for row in result['per_query']:
    if row['query_id'] == query_id:
        print(json.dumps(row, ensure_ascii=False, indent=2))

## Adapt the evaluation

Replace `corpus.json`, `queries.json` and `qrels.json` with your own permitted, explicitly judged records. Every candidate needs a grade for each query. Keep judgments out of ranker inputs. Add another ranking function and call `metrics(candidate_ids, judgments, k)` to evaluate it.

Grades are 0 irrelevant, 1 acceptable and 2 preferred in this authored dataset. Two-bedroom results are preferred to the acceptable three-bedroom result for the at-least-two query; the low-budget metro query prefers the two-bedroom candidate to a one-bedroom option. These are disclosed synthetic author preferences. They do not represent all users.

Run `python -m unittest -v test_search_eval` to execute 15 checks, including hand-calculated DCG, missing results, duplicate/unjudged identifiers, Unicode normalization, unknown policy, explicit negation and empty-result cases. Small synthetic examples are regression fixtures, not representative accuracy evidence. No telemetry or external ranking service is used.
